# 01 · Configuration and data adapter — Neonatal mortality NCHS (M4)

| Field | Content |
|---|---|
| Framework sequence | **01: S1 · S2 · S3** → 02: S4 · G1 → 03: S5 · S6 · S7 · G2 · G3 → 04: S8 → 05: S8 · S9 · G4 → 06: S10 · S11 |
| What it does | Builds the 8 birth cohorts 2016–2023 from the NCHS *Period/Cohort Linked Birth/Infant Death* public-use files: births of year N joined with the deaths in years N and N+1 of infants born in N |
| Stages | S1 (governance and configuration) · S2 (data adapter, universe and population) · S3 (labels) |
| Inputs | NCHS ZIP files in `../NCHS_data/raw/` (outside the repository) |
| Outputs | `data/interim/cohort_<year>.parquet` (not versioned) and `data/outputs_01_build_cohorts.txt` |
| Time and memory | About 2 minutes and 3 GB per cohort; cohorts already built are skipped |
| Next | `02_count_audit.ipynb` |
| Spanish version | `01_build_cohorts_es.ipynb` (same code) |

> **Rebuilding the Parquet files.** Cohorts that already exist in `data/interim/` are skipped. To build them again (for example, if the NCHS files or `libraries/nchs_reader.py` change), set **`REBUILD = True`** in the first code cell and run the notebook again. The contract table (notebook 03) must then be rebuilt too.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from libraries.nchs_reader import build_cohort, COHORTS, OUT_DIR

# Language of the outputs: "es" in the Spanish notebook, "en" in the English one (the only line that differs).
LANG = "en"
import importlib
import libraries.i18n_es, libraries.i18n          # reloaded so that a running kernel picks up dictionary changes
importlib.reload(libraries.i18n_es); importlib.reload(libraries.i18n)
from libraries.i18n import translate
pd.set_option("display.max_colwidth", None)      # decision texts are shown in full
import warnings
# scikit-learn >= 1.8 announces that LogisticRegression(penalty=...) will be replaced by l1_ratio. The models keep
# penalty (the hyperparameters chosen in notebook 04) so that results stay exactly as reported; the notice is hidden.
warnings.filterwarnings("ignore", message=".*'penalty' was deprecated.*", category=FutureWarning)
lines_es = []
if LANG == "es":                       # progress messages (print) also in Spanish
    import builtins
    def print(*args, **kwargs):
        builtins.print(*[translate(a, "es") if isinstance(a, str) and not a.startswith("\n== ") else a for a in args], **kwargs)

def write_outputs(path, header):
    # English text output (cited by the documents) and, from the Spanish notebook, a Spanish copy (*_es.txt).
    path.write_text(header + "\n" + "\n".join(lines) + "\n", encoding="utf-8")
    if LANG == "es":
        path.with_name(path.stem + "_es.txt").write_text(translate(header, "es") + "\n" + "\n".join(lines_es) + "\n", encoding="utf-8")


REBUILD = False      # True: rebuild every cohort even if its Parquet file exists
OUT_FILE = ROOT / "data" / "outputs_01_build_cohorts.txt"

## E1 · Governance and configuration — source, data use agreement, seed, software and publication rule

Fixed before running any analysis: the data source and its version, the data use agreement, the random seed, the software versions, the reference counts to be reproduced and the publication rule. The cell shows that configuration and checks that the files and packages are present.

| Result | What is done | In this model |
|---|---|---|
| The 8 NCHS ZIP files and the packages are present (ok) | On to S2 | 8 of 8 |
| A ZIP file is missing | Download it from the NCHS page; if the cohort is already in `data/interim/` it is not needed to go on | — |
| A package is missing | `pip install -r requirements.txt` with the virtual environment active | — |

In [ ]:
# Stage S1 check: the configuration every notebook of this repository uses, shown so it can be audited.
import platform, importlib
from libraries.nchs_reader import RAW_DIR
def version(module):
    try:
        return importlib.import_module(module).__version__
    except Exception:
        return "NOT INSTALLED"
zips_ok = [(RAW_DIR / f"{y + 1}PE{y}CO.zip").exists() for y in COHORTS]
rows = [("data source", "NCHS (CDC) Period/Cohort Linked Birth/Infant Death public-use files, releases 2017/2016 to 2024/2023", "fixed"),
        ("raw files", f"{sum(zips_ok)} of {len(COHORTS)} ZIP files in {RAW_DIR}",
         "ok" if all(zips_ok) else "MISSING: download from https://www.cdc.gov/nchs/data_access/vitalstatsonline.htm"),
        ("data use", "NCHS data use agreement: statistical analysis only; no re-identification; no linkage to identifiable data; data not redistributed", "rule"),
        ("publication rule", "only aggregate outputs (data/outputs_*.txt); counts 1-9 shown as <10; record-level files stay in data/ (not versioned)", "rule"),
        ("random seed", "20261123 (notebooks 03 to 06)", "fixed"),
        ("reference counts", "written by this notebook (data/outputs_01_build_cohorts.txt) and reproduced by notebook 02 (gate G1)", "fixed"),
        ("python", platform.python_version(), "info")]
rows += [(m, version(m), "ok" if version(m) != "NOT INSTALLED" else "INSTALL (requirements.txt)")
         for m in ["pandas", "numpy", "pyarrow", "sklearn", "xgboost", "matplotlib"]]
config = pd.DataFrame(rows, columns=["item", "value", "status"]).set_index("item")
translate(config, LANG)

## E2 · Universe and population — build the birth cohorts

Reads each file directly from the ZIP, joins births with deaths by cohort sequence number and year of death, and writes one Parquet per cohort with the original codes and the labels.

| Result | What is done | In this model |
|---|---|---|
| At least 99% of the deaths join their birth | Continue: the cohort is written | 100% in the 8 cohorts |
| Less than 99% | The adapter stops with an error: review the linkage keys | — |
| Cohort already built in an earlier run | It is skipped; the linkage rate is shown in the next section | — |

In [ ]:
summaries = []
for y in COHORTS:
    if REBUILD or not (OUT_DIR / f"cohort_{y}.parquet").exists():
        summaries.append(build_cohort(y))          # about 2 minutes and 3 GB of memory per cohort
    else:
        print(f"cohort {y}: already built")

## E2 · Universe and population — check of the linkage of deaths to their birth

For each cohort, counts the infant deaths of US-resident mothers in the death files and how many were joined to their birth in the Parquet. The cell stops (`assert`) if any cohort falls below 99%.

| Result | What is done | In this model |
|---|---|---|
| % joined ≥ 99% in every cohort | Passes: the death labels are complete | 100% in the 8 cohorts |
| < 99% in any cohort | It stops: review the linkage keys (cohort sequence number and year of death) | — |

In [ ]:
# Linkage check: share of each cohort's infant deaths joined to their birth. build_cohort() already stops
# with an error below 99%; this cell shows the rate for every cohort, also for cohorts built in an earlier
# run, by reading only the death (numerator) files of the ZIP (a few seconds per cohort).
import zipfile
import numpy as np
from libraries.nchs_reader import cohort_members, _read_member, DEATH_FIELDS
link = []
for y in COHORTS:
    zpath, _, nums = cohort_members(y)
    with zipfile.ZipFile(zpath) as zf:
        deaths = pd.concat([_read_member(zf, m, DEATH_FIELDS) for m in nums], ignore_index=True)
    deaths = deaths[(deaths["DOB_YY"] == y) & (deaths["RESTATUS"] < 4)].dropna(subset=["CO_SEQNUM", "CO_YOD"])
    linked = int(pd.read_parquet(OUT_DIR / f"cohort_{y}.parquet", columns=["died_infant"])["died_infant"].sum())
    link.append({"cohort": y, "infant deaths of the cohort (US residents)": len(deaths),
                 "joined to their birth": linked, "% joined": round(100 * linked / len(deaths), 2)})
link = pd.DataFrame(link).set_index("cohort")
link["check (>= 99%)"] = np.where(link["% joined"] >= 99, "passes", "FAILS")
assert (link["% joined"] >= 99).all(), "fewer than 99% of the deaths joined their birth: review the linkage keys"
translate(link, LANG)

## E3 · Outcome definition — reference counts per cohort

Population: live births to US-resident mothers at 22 to 47 weeks. Main label: death at 0 to 27 days; the sensitivity labels are also counted.

| Result | What is done | In this model |
|---|---|---|
| Counts written to `outputs_01_build_cohorts.txt` | They are the gate-G1 reference counts: the following notebooks reproduce them (`EXPECTED`) and stop if they differ | 29,829,867 live births; 82,559 neonatal deaths |

In [ ]:
rows = []
for y in COHORTS:
    d = pd.read_parquet(OUT_DIR / f"cohort_{y}.parquet",
                        columns=["RESTATUS", "COMBGEST", "CO_SEQNUM", "died_infant", "label_main",
                                 "label_early_neonatal", "died_first_24h", "label_day1_27"])
    us = d["RESTATUS"] < 4
    ga = d["COMBGEST"]
    pop = us & ga.between(22, 47).fillna(False)
    p28 = us & ga.between(28, 47).fillna(False)
    rows.append({
        "cohort": y, "births": len(d), "births_us_resident": int(us.sum()),
        "infant_deaths_linked": int(d["died_infant"].sum()),
        "ga_unknown": int((us & (ga == 99)).sum()), "ga_lt22": int((us & (ga < 22)).sum()),
        "pop_ge22wk": int(pop.sum()),
        "events_main_pop": int(d.loc[pop, "label_main"].sum()),
        "events_first24h_pop": int(d.loc[pop, "died_first_24h"].sum()),
        "events_day1_27_pop": int(d.loc[pop, "label_day1_27"].sum()),
        "events_early_neonatal_pop": int(d.loc[pop, "label_early_neonatal"].sum()),
        "pop_ge28wk": int(p28.sum()), "events_main_ge28": int(d.loc[p28, "label_main"].sum()),
    })
t = pd.DataFrame(rows).set_index("cohort").T
lines = ["Data adapter: birth cohorts built from the NCHS Period/Cohort Linked Birth/Infant Death "
         "public-use files (aggregate counts only)",
         "pop = live births to US-resident mothers with combined gestation 22-47 weeks; "
         "main label = neonatal death at 0-27 days", ""] + t.to_string().splitlines() + \
        ["", "Linkage of infant deaths to their birth (the adapter stops below 99%)"] + link.to_string().splitlines()
OUT_FILE.write_text("\n".join(lines) + "\n", encoding="utf-8")
if LANG == "es":
    lines_es = [translate(x, "es") for x in lines[:3]] + translate(t, "es").to_string().splitlines() + \
               ["", translate("Linkage of infant deaths to their birth (the adapter stops below 99%)", "es")] + translate(link, "es").to_string().splitlines()
    OUT_FILE.with_name(OUT_FILE.stem + "_es.txt").write_text("\n".join(lines_es) + "\n", encoding="utf-8")
translate(t, LANG)

## Notebook summary

Totals of the 8 cohorts.

In [ ]:
# Summary of notebook 01
summary = pd.Series({
    "cohorts": f"{min(COHORTS)}-{max(COHORTS)}",
    "infant deaths joined to their birth (lowest cohort, %)": float(link["% joined"].min()),
    "population (US residents, 22-47 weeks)": int(t.loc["pop_ge22wk"].sum()),
    "neonatal deaths 0-27 days": int(t.loc["events_main_pop"].sum()),
    "deaths per 1,000 live births": round(1000 * t.loc["events_main_pop"].sum() / t.loc["pop_ge22wk"].sum(), 2),
    "early neonatal deaths 0-6 days": int(t.loc["events_early_neonatal_pop"].sum()),
    "neonatal deaths at 28 weeks or more": int(t.loc["events_main_ge28"].sum())}, name="value")
translate(summary.to_frame(), LANG)